# EAIOS M8 · chat UI on the real Qwen gateway (browser test)

Choose **Runtime → Change runtime type → T4 GPU**, then run the cells in order.

Everything runs **inside this runtime over loopback**: the pinned model gateway (Qwen3-8B-AWQ, BGE-M3, development checker) on 127.0.0.1:8100, PostgreSQL/Redis/Qdrant, the backend API (`MODEL_MODE=gateway`) on 127.0.0.1:18000, the built web UI on 127.0.0.1:13000 and a headless Chromium (Playwright) that uses the UI like a person. No tunnel is opened and no port is exposed outside the runtime. Data is the synthetic NileTech fixture set.

The browser test uses **automatic routing**: there is no source picker, the server routes every question, and each answer must carry the label of the handler that replied. It passes only if:
1. the header badge reads **Connected · Qwen3-8B-AWQ** and the composer has no Source control;
2. "hello" and a fresh open-ended question get real replies labelled **General**, without citations;
3. a fresh annual-leave question is answered by **Documents** with a citation that opens the passage (*text hash verified*);
4. a sales question is answered by the **Sales report (SQL)** with a verified table and the **Email this report for approval** action (nothing is sent);
5. a vague request ("the numbers?") gets one short **clarifying question** and no run;
6. the chats survive a reload and a new sign-in under **Recents**, reopen with their answers, can be **deleted** (with confirmation, gone after reload), and a second employee cannot list, open or delete them;
7. the sales reader gets the no-sources refusal for the restricted finance question (routed to **Documents**), with no restricted figures on the page.

For the published answers the run record must show `inference_mode = gateway`, the pinned `Qwen/Qwen3-8B-AWQ@4da05a8edb55c6046cce958586c33b61da07bb79`, and an actual model call (purpose `general` / `answer` / `sql`, measured duration and token count).

Cell 5 downloads **`m8-chat-evidence-<UTC stamp>.zip`** (secret-scanned). Return that file.

Cell 6 is optional and off by default: it connects the laptop chat UI to this gateway through a tunnel, which needs your decision first (see its notes).

In [ ]:
# @title 1 · Checkout the branch under review and verify the GPU
import os, subprocess
REPO = "/content/Enterprise-AI-OS"
BRANCH = "feature/m8-chat-routing-history"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, "https://github.com/youssefhatembahig6-alt/Enterprise-AI-OS", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "fetch", "-q", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO, "checkout", "-q", "FETCH_HEAD"], check=True)
subprocess.run(["git", "-C", REPO, "log", "-1", "--format=%H %s"], check=True)
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], check=True)

In [ ]:
# @title 2 · Install the pinned vLLM environment (about 5 minutes)
subprocess.run(["python", f"{REPO}/ml/colab-m8-chat/run_m8_chat_colab.py", "vllm-env"], check=True)

In [ ]:
# @title 3 · Launch the model gateway on loopback (waits until the model is served)
# The gateway token: the Colab Secret EAIOS_GATEWAY_TOKEN if you created one (stable across
# restarts; needed for the optional laptop connection in cell 6), else a random one-off token.
import secrets
try:
    from google.colab import userdata
    _token = userdata.get("EAIOS_GATEWAY_TOKEN")
except Exception:
    _token = None
os.environ["GATEWAY_TOKEN"] = _token or os.environ.get("GATEWAY_TOKEN") or secrets.token_urlsafe(32)
print("gateway token from Colab Secret" if _token else "one-off random gateway token (not printed)")
%run /content/Enterprise-AI-OS/ml/model-gateway/colab_launch.py

In [ ]:
# @title 4 · Start the app, build the UI, run the browser test, bundle evidence (about 20–30 minutes)
subprocess.run(["python", f"{REPO}/ml/colab-m8-chat/run_m8_chat_colab.py", "all"], check=True)

In [ ]:
# @title 5 · Download the secret-scanned evidence bundle
import glob
from google.colab import files
bundles = sorted(glob.glob("/content/m8-chat-evidence-*.zip"))
if not bundles:
    raise RuntimeError("No evidence bundle exists; inspect the output of cell 4.")
files.download(bundles[-1])

## 6 · Optional: use the real model from the laptop chat UI (needs your decision first)

Cells 1–5 keep everything on loopback inside this runtime. To type your own questions in the laptop UI (http://localhost:13000) against this gateway, the laptop backend needs a URL that reaches port 8100 here. This cell opens an **ngrok tunnel** to the gateway, the same approach as the AI-SQL-Analyst project.

**Decide first.** Colab's FAQ lists *"connecting to remote proxies"* among the activities restricted on all managed runtimes (paid and free). On free runtimes it also restricts *"bypassing the notebook UI to interact primarily via a web UI"*. It does not name tunnels, so whether a tunnel to this gateway is acceptable for your plan is your decision; the project roadmap keeps the Colab-only constraint until that decision is made explicitly. The cell refuses to run until you set `TUNNEL_PERMITTED_BY_OWNER = True`.

**Setup (once).** Add two Colab Secrets (🔑 in the sidebar) and allow this notebook to read them:
* `EAIOS_GATEWAY_TOKEN`: a long random string you also keep on the laptop (`python3 -c "import secrets; print(secrets.token_urlsafe(32))"`). Re-run cell 3 after adding it, so the gateway uses it.
* `NGROK_AUTHTOKEN`: your ngrok account's authtoken.

Every gateway route except `/health` needs the bearer token; `/health` returns model names and revisions only. On the laptop, run the command the cell prints (`scripts/m8_connect_gateway.py`); it keeps the token server-side.

In [ ]:
# @title 6 · (Optional) Open the tunnel and print the laptop connect command
TUNNEL_PERMITTED_BY_OWNER = False  # set True only after you decided a tunnel is acceptable for your Colab plan
if not TUNNEL_PERMITTED_BY_OWNER:
    raise SystemExit("Tunnel not opened: set TUNNEL_PERMITTED_BY_OWNER = True after deciding (see the text above).")
from google.colab import userdata
if userdata.get("EAIOS_GATEWAY_TOKEN") != os.environ.get("GATEWAY_TOKEN"):
    raise SystemExit("Add the Colab Secret EAIOS_GATEWAY_TOKEN and re-run cell 3 first.")
subprocess.run(["pip", "-q", "install", "pyngrok>=7.2,<8"], check=True)
from pyngrok import conf, ngrok
conf.get_default().auth_token = userdata.get("NGROK_AUTHTOKEN")
tunnel = ngrok.connect(8100, "http", bind_tls=True)
print("Gateway URL:", tunnel.public_url)
print("On the laptop, in the repository:")
print(f"  python3 scripts/m8_connect_gateway.py connect --url {tunnel.public_url}")
print("It asks for EAIOS_GATEWAY_TOKEN (or reads it from the environment) and never prints it.")

## 7 · Private chat preview: type your own questions on the real Qwen (no tunnel)

Runs the chat UI and its API on **one loopback port** in this runtime (127.0.0.1:13100) and opens it with Colab's own `serve_kernel_port_as_window`. The window is reachable only from your signed-in Colab session, and only while this runtime lives. No public URL, no ngrok, no remote proxy. Needs cells 1–3. Cell 4 is optional: this cell starts the services and API itself if they are not running.

Before it opens the window, the cell runs the same browser test (automatic routing) on that same origin: the **Connected · Qwen3-8B-AWQ** badge, "hello" and a **fresh General** question, a **fresh Documents** question about annual leave with an opening citation, a sales question answered by the verified SQL report, a clarifying question, saved chats (reload, reopen, delete, second employee refused) and the sales reader's refusal.

For the published answers, the run records must show `inference_mode = gateway`, the pinned revision `4da05a8…` and an actual model call. Evidence: **`m8-chat-preview-evidence-<UTC stamp>.zip`** (cell 8). In the window, sign in with the details the cell prints (synthetic development account of this runtime), type your question and press **Enter**: the label on each answer shows which handler replied, and your chats are saved under **Recents**. After a runtime restart, run cells 1–3 and 7 again (the runtime's database is new, so earlier chats are gone).

In [ ]:
# @title 7 · Start the private preview, check it, and open it for you
import json
subprocess.run(["python", f"{REPO}/ml/colab-m8-chat/run_m8_chat_colab.py", "preview"], check=True)
creds = json.load(open("/content/m8-run/secrets/dev-credentials.json"))["niletech"]
print("Sign in  ->  Organization: niletech   Email: finance.reader@niletech.example")
print("Password (synthetic development account of this runtime):", creds["finance.reader@niletech.example"])
from google.colab import output
output.serve_kernel_port_as_window(13100, anchor_text="Open the Enterprise AI OS chat (private, this session only)")

In [ ]:
# @title 8 · Download the private preview's evidence bundle
import glob
from google.colab import files
bundles = sorted(glob.glob("/content/m8-chat-preview-evidence-*.zip"))
if not bundles:
    raise RuntimeError("No preview evidence bundle exists; inspect the output of cell 7.")
files.download(bundles[-1])